# Pipeline SDC — Analyse de métadonnées par LLM

Ce notebook présente le pipeline `metadata-analysis-llm-for-sdc`.

Run pip install -r requirements-notebook.txt

In [ ]:
# pour changer de modèle (autre que qwen dans ce cas)
import os 
os.environ["LLM_MODEL"] = "gemma4-26b-moe"

## Prérequis. Mise en place

> **Prérequis Onyxia :** service VSCode-python lancé avec `CLE_API_OPENWEBUI` déclarée en variable d'environnement, et `pip install -r requirements.txt` exécuté dans le terminal.

In [ ]:
import os, sys, tempfile
from pathlib import Path
import s3fs
from IPython.display import Markdown, display

# Détection de la racine du repo (fonctionne que le notebook soit lancé
# depuis la racine ou depuis notebooks/)
for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / "core").is_dir():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Impossible de localiser 'core/'. Lancez Jupyter depuis la racine du repo.")

sys.path.insert(0, str(REPO_ROOT))
from core import pipeline, transform_output

# Vérification de l'environnement
api_key = os.environ.get("CLE_API_OPENWEBUI") or os.environ.get("OPENAI_API_KEY")
assert api_key, "Clé API introuvable — déclarez CLE_API_OPENWEBUI dans les variables d'environnement Onyxia."
print(f"✓ Clé API       : {'*' * (len(api_key))}")
print(f"✓ Modèle        : {os.environ.get('LLM_MODEL', 'qwen3-6-35b-moe')}")

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
)
print("✓ MinIO         : connexion initialisée")

## 1. Sérialisation — Lecture et transformation markdown

`transform_input.serialize` convertit le classeur en Markdown pur.


In [ ]:
INPUT_S3  = "path"
OUTPUT_S3 = "path" # le chemin du fichier de sortie, qui est attendu en csv

# Téléchargement depuis MinIO vers /tmp
tmp_input = Path(f"/tmp/input{Path(INPUT_S3).suffix}")
with fs.open(INPUT_S3, "rb") as f_in:
    tmp_input.write_bytes(f_in.read())
print(f"✓ Téléchargé : {INPUT_S3}\n")

# Sérialisation
md_input = pipeline.serialize(tmp_input)
print(f"  {len(md_input)} caractères — aperçu :")
display(Markdown(md_input))

## 2. Phase 1 — le modèle analyse et pose ses questions

`pipeline.start()` envoie le Markdown au modèle avec les instructions.

Le modèle produit une liste de questions

In [ ]:
print("Phase 1 — envoi au modèle...\n")
r = pipeline.start(md_input)

if r.auto_continued:
    print("Le modèle n'a posé aucune question — JSON produit directement.")
    print("Passez à la cellule Phase 2 (la cellule 'Réponses' sera ignorée).")
else:
    print("─" * 70)
    print(r.questions)
    print("─" * 70)
    print("\n→ Remplissez la variable `answers` dans la cellule ci-dessous, puis exécutez-la.")

## 3a. Réponses du producteur

Remplissez la variable `answers` avec vos réponses aux questions ci-dessus
(une réponse par question, dans l'ordre), puis exécutez la cellule.


In [ ]:
# Remplissez vos réponses ici avant d'exécuter cette cellule.
answers = """
"""

print("Réponses enregistrées — exécutez la cellule Phase 2.")

## 3b. — JSON, validation et CSV final

`pipeline.answer()` envoie les réponses au modèle qui produit le JSON.
Le JSON est ensuite **validé contre le schéma**, puis
converti en tableau plat et écrit sur MinIO.


In [ ]:
import pandas as pd

if r.auto_continued:
    # Phase 1 a déjà produit le JSON — pas besoin d'un second appel
    records = r.records
    print("Résultats issus de la Phase 1 (aucune question posée).\n")
else:
    print("Phase 2 — envoi des réponses au modèle...\n")
    records = pipeline.answer(r.history, answers)
    print(f"✓ {len(records)} enregistrement(s) validés contre le schéma.\n")

# Écriture du CSV sur MinIO
tmp_csv = Path("/tmp/output.csv")
cols, rows = transform_output.write_csv(records, tmp_csv)

with open(tmp_csv, "r", encoding="utf-8-sig") as f:
    csv_content = f.read()
with fs.open(OUTPUT_S3, "w", encoding="utf-8") as f_out:
    f_out.write(csv_content)

print(f"✓ CSV écrit sur MinIO : {OUTPUT_S3}")
print(f"  {len(rows)} lignes × {len(cols)} colonnes\n")

# Affichage du résultat
df = pd.read_csv(tmp_csv, encoding="utf-8-sig", keep_default_na=False)
display(df)

## 6. Suivi des erreurs — `results.csv`

Compare la sortie du pipeline avec un **CSV de référence** (sortie attendue corrigée à la main)
et enregistre les résultats dans `notebook/results.csv`.

Chaque run ajoute **une ligne** avec :
- le nom du fichier d'entrée et l'horodatage UTC du run,
- le nombre total de cellules incorrectes,
- la décomposition par variable (`field`, `hrc_field`, `indicator`, `hrc_indicator`, variables de croisement).
- la différence entre le nombre de tableaux de la correction et du output du LLM

Répétés sur plusieurs fichiers, ces chiffres permettent d'identifier si le LLM se trompe
**systématiquement** sur une variable précise (problème de prompt/modèle) plutôt qu'aléatoirement.

> **Avant d'exécuter** : renseignez `REFERENCE_CSV` avec le chemin vers votre CSV de référence
> (MinIO ou local). Le CSV de référence doit avoir la même structure que la sortie du pipeline.

In [ ]:
import re, csv
from datetime import datetime, timezone

# ── À remplir : chemin vers le CSV de référence (sortie attendue pour ce fichier) ──
REFERENCE_CSV = "(path_here)"   # MinIO (ex. user/data/ref.csv) ou chemin local
# ────────────────────────────────────────────────────────────────────────────────────

RESULTS_CSV  = REPO_ROOT / "notebook" / "results.csv"
RESULTS_HTML = REPO_ROOT / "notebook" / "results.html"

def _load_ref(path, fs):
    try:
        with fs.open(path, "r", encoding="utf-8-sig") as f:
            return pd.read_csv(f, keep_default_na=False)
    except Exception:
        return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)

ref = _load_ref(REFERENCE_CSV, fs)
out = df.copy()

# ── Comparaison du nombre de tableaux ───────────────────────────────────────
n_tables_ref  = len(ref)
n_tables_out  = len(out)
n_tables_diff = n_tables_out - n_tables_ref

# ── Alignement sur table_name (fallback positionnel) ────────────────────────
def _align(ref_df, out_df):
    if "table_name" in ref_df.columns and "table_name" in out_df.columns:
        ref_ids = ref_df["table_name"].tolist()
        out_lookup = set(out_df["table_name"])
        common = [t for t in ref_ids if t in out_lookup]
        if common:
            ra = ref_df.set_index("table_name").reindex(common)
            oa = out_df.set_index("table_name").reindex(common)
            return ra, oa, len(ref_df) - len(common), len(out_df) - len(common)
    n = min(len(ref_df), len(out_df))
    return (ref_df.iloc[:n].reset_index(drop=True),
            out_df.iloc[:n].reset_index(drop=True),
            len(ref_df) - n, len(out_df) - n)

ref_a, out_a, n_miss_ref, n_miss_out = _align(ref, out)

# Normalise les cellules vides en "NA" dans la référence (sentinel du schéma)
ref_a = ref_a.fillna("NA").replace("", "NA")
out_a = out_a.fillna("NA").replace("", "NA")

# ── Comptage des cellules incorrectes par colonne ────────────────────────────
SPAN_RE     = re.compile(r"^spanning_\d+$")
HRC_SPAN_RE = re.compile(r"^hrc_spanning_\d+$")

col_errors = {"table_name": 0, "field": 0, "hrc_field": 0,
              "indicator": 0, "hrc_indicator": 0,
              "spanning": 0, "hrc_spanning": 0}

for col in ref_a.columns:
    if col not in out_a.columns:
        continue
    n_wrong = int((ref_a[col].astype(str) != out_a[col].astype(str)).sum())
    if col in col_errors:
        col_errors[col] += n_wrong
    elif SPAN_RE.match(col):
        col_errors["spanning"] += n_wrong
    elif HRC_SPAN_RE.match(col):
        col_errors["hrc_spanning"] += n_wrong

unmatched_err = (n_miss_ref + n_miss_out) * max(len(ref_a.columns), 1)
total_wrong   = sum(col_errors.values()) + unmatched_err

# ── Résumé ───────────────────────────────────────────────────────────────────
diff_label = "=" if n_tables_diff == 0 else f"{n_tables_diff:+d}"
print(f"  Tableaux attendus  : {n_tables_ref}")
print(f"  Tableaux produits  : {n_tables_out}  [{diff_label}]")
if n_tables_diff != 0:
    print(f"  !! LLM a {'ajouté' if n_tables_diff > 0 else 'omis'} {abs(n_tables_diff)} tableau(x)")
print(f"\n  {'Colonne':<22} {'Cellules incorrectes':>20}")
print("  " + "─" * 44)
for k, v in col_errors.items():
    print(f"  {k:<22} {v:>20}")
if n_miss_ref or n_miss_out:
    print(f"\n  Lignes manquantes : {n_miss_ref}")
    print(f"  Lignes en excès   : {n_miss_out}")
print(f"\n  ➜  TOTAL : {total_wrong} cellules incorrectes")

# ── results.csv ──────────────────────────────────────────────────────────────
RESULT_COLS = [
    "filename", "timestamp", "total_wrong_cells",
    "n_tables_ref", "n_tables_out", "n_tables_diff",
    "table_name", "field", "hrc_field",
    "indicator", "hrc_indicator",
    "spanning", "hrc_spanning",
    "missing_from_output", "extra_in_output",
]
row = {
    "filename":            Path(INPUT_S3).name,
    "timestamp":           datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "total_wrong_cells":   total_wrong,
    "n_tables_ref":        n_tables_ref,
    "n_tables_out":        n_tables_out,
    "n_tables_diff":       n_tables_diff,
    **col_errors,
    "missing_from_output": n_miss_ref,
    "extra_in_output":     n_miss_out,
}

write_header = not RESULTS_CSV.exists()
with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=RESULT_COLS, extrasaction="ignore")
    if write_header:
        w.writeheader()
    w.writerow(row)